# B06 — Thử nghiệm đối chứng để giải thích lỗi albopictus

**Đọc kết quả 05 trước khi diễn giải.** Giữ nguyên A_v1 và DS-CNN của B. Năm cấu hình cùng trần ngân sách, mỗi cấu hình chỉ đổi **một yếu tố** so với baseline mới:

| Cấu hình | Thay đổi | Câu hỏi / giới hạn |
|---|---|---|
| baseline | Log-mel + mean/std train, natural sampling, CE thường | Mốc đối chứng chạy cùng code/seed với các thử nghiệm; không mặc định tái lập đúng số của 04 |
| rms_per_window | Cân RMS từng window về 0,02, giới hạn tăng gain 30 dB; rồi fit mean/std train | Mức âm có ảnh hưởng không? Không khôi phục SNR hoặc bù đáp ứng microphone |
| highpass_150 | Butterworth bậc 2, 150 Hz, causal trên cả clip trước khi cắt window | Dải thấp có ảnh hưởng không? Có thể bỏ thông tin sinh học; một cutoff thăm dò, chưa tối ưu |
| weighted_ce | Trọng số CE nghịch đảo số window train | Thiếu đóng góp của lớp nhỏ có ảnh hưởng không? Không thêm độ đa dạng nguồn |
| balanced_sampler | Lấy mẫu có hoàn lại nghịch đảo số window/lớp | Cân số lượt tiếp xúc lớp có giúp không? Lặp lại dữ liệu, không tạo thông tin mới |

Không kết hợp các thay đổi trong lần này. Tất cả thống kê fit trên train của từng frontend; cùng phép biến đổi áp dụng validation. Ghi cả clip/window macro-F1, recall/F1 từng loài, số update và thời gian. Chọn checkpoint bằng **validation clip macro-F1**, tie giữ epoch sớm, giống tiêu chí B.

**Smoke**: 1 seed, 2 epoch × tối đa 2 batch mỗi cấu hình; chỉ kiểm luồng, không xếp hạng. **Full**: mặc định 3 seed, tối đa 40 epoch, patience 8. Không đọc/scoring test. Lựa chọn qua validation có thể overfit chính nguồn D1 này; chưa chứng minh dùng tốt trên INMP441 hay nguồn mới.

## 0. Thiết lập và chuẩn bị môi trường
Sửa tham số trực tiếp ở cell dưới. Colab mount shortcut `Edge_AI_Mosquito`, clone/pull repo và chỉ phục hồi data/A từ bundle đã kiểm checksum. Local dùng dữ liệu trong project hoặc đường dẫn raw đã ghi trong manifest. Không cần cài thêm thư viện ngoài môi trường B.

In [ ]:
from pathlib import Path, PureWindowsPath, PurePosixPath
import os, sys, subprocess, shutil, csv, hashlib, io, json, re, time, zipfile
import importlib.util, importlib.metadata
EXECUTION_ENV='auto'  # auto / local / colab
DRIVE_FOLDER=Path('/content/drive/MyDrive/Edge_AI_Mosquito')
COLAB_PROJECT_DIR=Path('/content/Edge_AI_Mosquito')
REPO_URL='https://github.com/ancaranovik/Edge_AI_Mosquito.git';REPO_BRANCH='main'
BUNDLE_SHA256='29a099018450a1dc1e54201c58d441a72d53ec0c8ace10b8921b40c4c34c6278'
SAVE_TO_DRIVE=True
CPU_THREADS=4
NOTEBOOK_NAME='06_B_controlled_experiments.ipynb'
STAGE='controlled_experiments'
RUN_MODE='smoke'
# Smoke kiểm code; đổi full để huấn luyện toàn bộ pool train/validation.
EXPERIMENTS=['baseline','rms_per_window','highpass_150','weighted_ce','balanced_sampler']
FULL_SEEDS=[42,43,44]
SELECTION_SEED=42
DEVICE_CHOICE='auto'  # auto / cuda / cpu
REQUIRE_COLAB_GPU=True
BATCH_SIZE=64;MAX_EPOCHS=40;PATIENCE=8;LEARNING_RATE=.001
SMOKE_TRAIN_CLIPS_PER_CLASS=4;SMOKE_VAL_CLIPS_PER_CLASS=2;SMOKE_WINDOWS_PER_CLIP=2
SMOKE_BATCH_SIZE=8;SMOKE_EPOCHS=2;SMOKE_MAX_BATCHES=2
RMS_TARGET=.02;MAX_RMS_GAIN_DB=30.
HIGHPASS_HZ=150.;HIGHPASS_ORDER=2

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
assert EXECUTION_ENV in ('auto','local','colab') and RUN_MODE in ('smoke','full')
try:has_colab=importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:has_colab=False
IS_COLAB=EXECUTION_ENV=='colab' or (EXECUTION_ENV=='auto' and has_colab and Path('/content').is_dir())
print('Môi trường:', 'Colab' if IS_COLAB else 'local','| Mode:',RUN_MODE)

In [ ]:
def bootstrap_sha(path):
    with Path(path).open('rb') as f: return hashlib.file_digest(f,'sha256').hexdigest()

def archive_relative(name):
    rel=PurePosixPath(name)
    if not name or rel.is_absolute() or '..' in rel.parts or chr(92) in name or ':' in name:
        raise ValueError('Đường dẫn archive không hợp lệ: '+name)
    return rel

def extract_dataset_bundle(archive,project_root):
    project_root=Path(project_root).resolve()
    with zipfile.ZipFile(archive) as z:
        names=z.namelist()
        assert len(names)==len(set(names)), 'Archive trùng tên file'
        for name in names: archive_relative(name)
        rows=list(csv.DictReader(io.StringIO(z.read('bundle_inventory.csv').decode('utf-8-sig'))))
        selected=[r for r in rows if r['archive_path'].startswith(('data/','results/A/'))]
        assert len({r['archive_path'] for r in selected})==len(selected)
        assert {n for n in names if n.startswith(('data/','results/A/'))}=={r['archive_path'] for r in selected}
        assert sum(r['type']=='audio' for r in selected)==2905
        needed=sum(int(r['bytes']) for r in selected if not project_root.joinpath(*archive_relative(r['archive_path']).parts).exists())
        if shutil.disk_usage(project_root).free < needed+100_000_000:
            raise OSError('Ổ tạm không đủ chỗ giải nén gói A_v1')
        reused=written=0
        for i,row in enumerate(selected,1):
            rel=archive_relative(row['archive_path']);target=project_root.joinpath(*rel.parts)
            assert target.resolve().is_relative_to(project_root)
            if target.is_file() and target.stat().st_size==int(row['bytes']) and bootstrap_sha(target)==row['sha256']:
                reused+=1;continue
            target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_name(target.name+'.extracting')
            h=hashlib.sha256();size=0
            with z.open(row['archive_path']) as src,tmp.open('wb') as dst:
                while chunk:=src.read(1024*1024): dst.write(chunk);h.update(chunk);size+=len(chunk)
            if h.hexdigest()!=row['sha256'] or size!=int(row['bytes']):
                raise ValueError('Hash giải nén sai: '+str(rel))
            os.replace(tmp,target);written+=1
            if i%500==0: print('Giải nén/kiểm tra:',i,'/',len(selected),flush=True)
    return dict(bundle_files=len(selected),audio_files=2905,files_written=written,files_reused=reused)

bootstrap_info=dict(environment='colab' if IS_COLAB else 'local',bundle_sha256='not_used')
if IS_COLAB:
    if not (DRIVE_FOLDER/'A_v1_bundle.zip').is_file():
        from google.colab import drive
        drive.mount('/content/drive')
    source_bundle=DRIVE_FOLDER/'A_v1_bundle.zip'
    if not source_bundle.is_file():
        raise FileNotFoundError(f'Không thấy {source_bundle}. Kiểm tra tên shortcut trong My Drive A.')
    ROOT=COLAB_PROJECT_DIR.resolve()
    if not (ROOT/'.git').exists():
        if ROOT.exists() and any(ROOT.iterdir()):
            raise RuntimeError('Folder Colab đã có file nhưng chưa có Git; chọn COLAB_PROJECT_DIR khác.')
        subprocess.run(['git','clone','--branch',REPO_BRANCH,REPO_URL,str(ROOT)],check=True)
    else:
        remote=subprocess.check_output(['git','-C',str(ROOT),'remote','get-url','origin'],text=True).strip()
        assert remote.rstrip('/').removesuffix('.git')==REPO_URL.rstrip('/').removesuffix('.git')
        dirty=subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'],text=True)
        if dirty.strip(): raise RuntimeError('Code trên runtime đã sửa. Giữ bản sửa trước khi pull; không tự reset.')
        branch=subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'],text=True).strip()
        assert branch==REPO_BRANCH,'Checkout branch khác; không tự đổi hoặc ghi đè.'
        subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin',REPO_BRANCH],check=True)
    staging=ROOT/'data/.bundle';staging.mkdir(parents=True,exist_ok=True)
    local_bundle=staging/'A_v1_bundle.zip'
    if not local_bundle.is_file() or bootstrap_sha(local_bundle)!=BUNDLE_SHA256:
        if shutil.disk_usage(ROOT).free < source_bundle.stat().st_size+100_000_000:
            raise OSError('Ổ tạm không đủ chỗ copy archive')
        tmp=local_bundle.with_suffix('.zip.copying')
        shutil.copyfile(source_bundle,tmp)
        assert bootstrap_sha(tmp)==BUNDLE_SHA256,'Archive trên Drive chưa khớp gói A_v1 đã chốt'
        os.replace(tmp,local_bundle)
    bootstrap_info.update(extract_dataset_bundle(local_bundle,ROOT),bundle_sha256=BUNDLE_SHA256)
    os.chdir(ROOT)
else:
    ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks').is_dir() and (p/'README.md').is_file()),None)
    if ROOT is None: raise FileNotFoundError('Mở notebook từ folder repo local.')
    ROOT=ROOT.resolve()
try:
    bootstrap_info['git_commit']=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
except subprocess.CalledProcessError:
    bootstrap_info['git_commit']='not_available'
print('Project trên kernel:',ROOT)
print(bootstrap_info)

In [ ]:
packages={'numpy':'numpy','pandas':'pandas','scipy':'scipy','soundfile':'soundfile','matplotlib':'matplotlib','seaborn':'seaborn','scikit-learn':'sklearn','nbformat':'nbformat','torch':'torch'}
missing=[p for p,m in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    if not IS_COLAB:raise ImportError('Thiếu thư viện: '+', '.join(missing)+'. Dùng .venv/requirements.txt.')
    if 'torch' in missing:raise ImportError('Chọn runtime Colab có PyTorch; không cài bản CPU từ requirements vào Colab.')
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)

## 1. Hàm dùng chung và xác minh A_v1
Các cell thuật toán phải khớp source trên kernel (quan trọng khi dùng VS Code nối Colab). Giữ nhãn, split và biên window của A. Loader từ chối audio test trước khi truy cập file; resample/canonical hash phải khớp A.

In [ ]:
from math import gcd
from functools import lru_cache
from datetime import datetime
import copy, html, platform, random
import nbformat
import numpy as np
import pandas as pd
import scipy
from scipy import signal
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from IPython.display import display, Markdown, Audio
LABEL_NAMES=['Aedes aegypti','Aedes albopictus','Anopheles dirus','Anopheles minimus']
LABELS=np.arange(4)
SAMPLE_RATE=8000; WINDOW_SAMPLES=2400; STFT_N=256; STFT_HOP=80; MEL_BINS=40
PSD_FLOOR=1e-12
versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,
              scipy=scipy.__version__,soundfile=sf.__version__,sklearn=sklearn.__version__,torch=str(torch.__version__))
sns.set_theme(style='whitegrid');plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
def sha(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
NOTEBOOK_PATH=ROOT/'notebooks'/NOTEBOOK_NAME
notebook_source=nbformat.read(NOTEBOOK_PATH,as_version=4)
core_cells={c.metadata.tags[0]:c.source for c in notebook_source.cells if c.cell_type=='code' and c.metadata.get('tags') and c.metadata.tags[0] not in ('settings','bootstrap','dependencies')}
CODE_SHA=digest(core_cells); verified_tags=set()
def verify_cell(tag):
    from IPython import get_ipython
    shell=get_ipython()
    if shell is not None and shell.history_manager.input_hist_raw:
        actual=shell.history_manager.input_hist_raw[-1].replace('\r\n','\n').strip()
        if actual!=core_cells[tag].replace('\r\n','\n').strip():
            raise RuntimeError('Cell khác source trên kernel: '+tag+'. Save/push/pull notebook trước khi chạy; tham số được sửa ở cell đầu.')
    verified_tags.add(tag)
def atomic_bytes(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
    tmp.write_bytes(value)
    for attempt in range(6):
        try:os.replace(tmp,path);return
        except PermissionError:
            if attempt==5:raise
            time.sleep(.05*2**attempt)
def save_table(frame,name):
    atomic_bytes(TABLES/(name+'.csv'),frame.to_csv(index=False).encode('utf-8-sig'));return frame
def save_figure(name):
    plt.tight_layout();plt.savefig(FIGURES/(name+'.png'),dpi=150,bbox_inches='tight');plt.show();plt.close()
checks=[]
def check(name,condition):
    checks.append(dict(check=name,passed=bool(condition)))
    if not condition:raise AssertionError(name)
CLIP_PATH=ROOT/'data/manifests/biodcase_four_species_clips.csv'
WINDOW_PATH=ROOT/'data/manifests/biodcase_four_species_windows.csv'
FREEZE_PATH=ROOT/'results/A/preparation/tables/dataset_freeze.csv'
PROTECTED={p:sha(p) for p in [CLIP_PATH,WINDOW_PATH,FREEZE_PATH,ROOT/'notebooks/03_A_data_preparation.ipynb',ROOT/'notebooks/04_B_baseline.ipynb']}
def init_run(settings):
    global RUN_ID,RUN_OUT,TABLES,FIGURES,CACHE,REMOTE
    settings=dict(settings,source_sha=CODE_SHA,versions=versions,freeze_sha=sha(FREEZE_PATH),
                  manifests=[sha(CLIP_PATH),sha(WINDOW_PATH)])
    RUN_ID=digest(settings);RUN_OUT=ROOT/'results/B'/STAGE/RUN_MODE/RUN_ID[:16]
    TABLES=RUN_OUT/'tables';FIGURES=RUN_OUT/'figures';CACHE=RUN_OUT/'cache'
    for folder in [TABLES,FIGURES,CACHE]:folder.mkdir(parents=True,exist_ok=True)
    REMOTE=DRIVE_FOLDER/'runs/B'/STAGE/RUN_MODE/RUN_ID[:16] if IS_COLAB and SAVE_TO_DRIVE else None
    save_table(pd.DataFrame([dict(item=k,value=json.dumps(v,ensure_ascii=False,default=str)) for k,v in settings.items()]+[dict(item='run_id',value=RUN_ID)]),'run_settings')
    atomic_bytes(RUN_OUT/'source_notebook.ipynb',nbformat.writes(notebook_source).encode('utf-8'))
    if REMOTE is not None:
        REMOTE.mkdir(parents=True,exist_ok=True);probe=REMOTE/'write_probe.tmp'
        probe.write_bytes(b'diagnostic write check');assert probe.read_bytes()==b'diagnostic write check';probe.unlink()
    print('Run:',RUN_ID[:16],'|',RUN_MODE,'| Output:',RUN_OUT)
    if REMOTE is not None:print('Drive:',REMOTE)
verify_cell('helpers')

In [ ]:
verify_cell('data')
def frozen_path(value):
    rel=PureWindowsPath(str(value));assert not rel.is_absolute() and not rel.drive and '..' not in rel.parts
    return ROOT.joinpath(*rel.parts)
freeze=pd.read_csv(FREEZE_PATH,keep_default_na=False).set_index('item').value.to_dict()
assert freeze['protocol_version']=='A_v1'
a_source=nbformat.read(ROOT/'notebooks/03_A_data_preparation.ipynb',as_version=4)
assert hashlib.sha256('\n'.join(c.source for c in a_source.cells).encode()).hexdigest()==freeze['notebook_source_sha256']
for key,value in freeze.items():
    if key.endswith('_path'):assert sha(frozen_path(value))==freeze[key[:-5]+'_sha256'],key
clips=pd.read_csv(CLIP_PATH,keep_default_na=False);windows=pd.read_csv(WINDOW_PATH)
kept=clips[clips.status.eq('keep')].copy()
assert clips.clip_id.is_unique and windows.window_id.is_unique
assert kept.groupby('leakage_group').split.nunique().le(1).all()
assert kept[['label_id','species']].drop_duplicates().set_index('label_id').species.to_dict()==dict(enumerate(LABEL_NAMES))
assert windows.end_sample.sub(windows.start_sample).eq(WINDOW_SAMPLES).all() and windows.sample_rate.eq(SAMPLE_RATE).all()
joined=windows.merge(kept[['clip_id','label_id','species','split','processed_frames']],on='clip_id',suffixes=('','_clip'),validate='many_to_one')
for col in ['label_id','species','split']:assert joined[col].eq(joined[col+'_clip']).all()
assert joined.end_sample.le(joined.processed_frames).all()
# Metadata test có trong manifest để kiểm tra split; audio lookup chỉ chứa train/validation.
allowed_windows=windows[windows.split.isin(['train','validation'])].copy()
test_ids=set(kept[kept.split.eq('test')].clip_id)
allowed_ids=set(allowed_windows.clip_id)
clip_lookup=kept[kept.clip_id.isin(allowed_ids)].set_index('clip_id').copy()
assert not allowed_ids&test_ids and set(clip_lookup.index)==allowed_ids
assert clips.path.map(lambda p:PureWindowsPath(p).name).is_unique
accessed=set()
def audio_path(row):
    candidate=ROOT/'data/audio/biodcase_A_v1'/PureWindowsPath(row.path).name
    if candidate.is_file():return candidate
    if not IS_COLAB and Path(row.path).is_file():return Path(row.path)
    raise FileNotFoundError('Không thấy WAV A_v1: '+str(candidate))
@lru_cache(maxsize=2)
def read_clip(clip_id):
    if clip_id not in allowed_ids or clip_id in test_ids:raise ValueError('Không được đọc audio ngoài train/validation: '+str(clip_id))
    row=clip_lookup.loc[clip_id];p=audio_path(row)
    assert sha(p)==row.sha256,'Raw WAV thay đổi: '+clip_id
    x,sr=sf.read(p,dtype='float64',always_2d=True)
    assert len(x)>0 and np.isfinite(x).all();x=x.mean(axis=1)
    if sr!=SAMPLE_RATE:
        factor=gcd(int(sr),SAMPLE_RATE);x=signal.resample_poly(x,SAMPLE_RATE//factor,int(sr)//factor)
    x=np.asarray(x,dtype='<f4')
    actual=hashlib.sha256(f'{SAMPLE_RATE}|1|{len(x)}|'.encode()+x.tobytes()).hexdigest()
    assert actual==row.canonical_sha256 and len(x)==int(row.processed_frames),'Canonical khác A: '+clip_id
    accessed.add(clip_id);return x
HANN=signal.windows.hann(STFT_N,sym=False);FREQUENCIES=np.fft.rfftfreq(STFT_N,1/SAMPLE_RATE)
mel_edges=700*(10**(np.linspace(0,2595*np.log10(1+4000/700),MEL_BINS+2)/2595)-1)
mel_filter=np.maximum(0,np.minimum((FREQUENCIES[None,:]-mel_edges[:-2,None])/(mel_edges[1:-1,None]-mel_edges[:-2,None]),
    (mel_edges[2:,None]-FREQUENCIES[None,:])/(mel_edges[2:,None]-mel_edges[1:-1,None])))
mel_filter/=mel_filter.sum(axis=1,keepdims=True)
def logmel(x):
    frames=np.lib.stride_tricks.sliding_window_view(np.asarray(x,dtype=np.float64),STFT_N)[::STFT_HOP]
    power=np.abs(np.fft.rfft(frames*HANN,axis=-1))**2/(SAMPLE_RATE*np.sum(HANN**2));power[:,1:-1]*=2
    out=10*np.log10(np.maximum(mel_filter@power.T,PSD_FLOOR))
    assert out.shape==(40,27) and np.isfinite(out).all();return out.astype('<f4')
def aggregate_clips(frame,p):
    p=np.asarray(p,dtype=np.float64)
    assert p.shape==(len(frame),4) and np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-5)
    tmp=frame[['clip_id','label_id','species','domain','leakage_group']].copy()
    for i in LABELS:tmp[f'p{i}']=p[:,i]
    assert tmp.groupby('clip_id').label_id.nunique().eq(1).all()
    c=tmp.groupby('clip_id',sort=True).agg(label_id=('label_id','first'),species=('species','first'),domain=('domain','first'),
        leakage_group=('leakage_group','first'),windows=('label_id','size'),**{f'p{i}':(f'p{i}','mean') for i in LABELS}).reset_index()
    c['prediction']=c[[f'p{i}' for i in LABELS]].to_numpy().argmax(1);return c
check('Freeze A_v1 và nhãn cố định',freeze['protocol_version']=='A_v1')
try:read_clip(sorted(test_ids)[0]);blocked=False
except ValueError:blocked=True
check('Từ chối audio test trước I/O',blocked and not accessed)
print('Train/validation:',len(allowed_windows),'window;',len(allowed_ids),'clip đủ dài. Không đọc audio test.')

## 2. Chọn dữ liệu, thiết bị và ghi cấu hình thử nghiệm
Mọi cấu hình dùng cùng ID window. Seed lấy mẫu smoke độc lập với seed model. Bản full dùng toàn bộ train/validation A_v1. Source, dữ liệu, frontend, môi trường CUDA và recipe đều có trong run identity để không resume nhầm.

In [ ]:
verify_cell('experiment_setup')
assert set(EXPERIMENTS)<=set(['baseline','rms_per_window','highpass_150','weighted_ce','balanced_sampler']) and 'baseline' in EXPERIMENTS
assert len(EXPERIMENTS)==len(set(EXPERIMENTS)) and len(FULL_SEEDS)==len(set(FULL_SEEDS)) and len(FULL_SEEDS)>0
assert DEVICE_CHOICE in ('auto','cuda','cpu')
assert CPU_THREADS>0 and BATCH_SIZE>0 and MAX_EPOCHS>0 and PATIENCE>0 and SMOKE_EPOCHS>0 and SMOKE_MAX_BATCHES>0
assert 0<RMS_TARGET<1 and 0<=MAX_RMS_GAIN_DB<=60 and 0<HIGHPASS_HZ<4000 and 1<=HIGHPASS_ORDER<=8
if (DEVICE_CHOICE=='cuda' or (IS_COLAB and REQUIRE_COLAB_GPU)) and not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU CUDA. Chọn Colab GPU; local CPU được dùng cho smoke.')
DEVICE=torch.device('cuda' if DEVICE_CHOICE=='cuda' or (DEVICE_CHOICE=='auto' and torch.cuda.is_available()) else 'cpu')
torch.set_num_threads(CPU_THREADS);torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
torch.backends.cudnn.allow_tf32=False;torch.backends.cuda.matmul.allow_tf32=False
compute=dict(device=DEVICE.type,cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),gpu=torch.cuda.get_device_name(DEVICE) if DEVICE.type=='cuda' else '',
    capability=list(torch.cuda.get_device_capability(DEVICE)) if DEVICE.type=='cuda' else [],cpu_threads=CPU_THREADS,float32_no_amp_no_tf32=True,deterministic=True)
seeds=FULL_SEEDS if RUN_MODE=='full' else FULL_SEEDS[:1]
def select_smoke(frame,nclips):
    chunks=[]
    for label,g in frame.groupby('label_id',sort=True):
        ids=np.asarray(sorted(g.clip_id.unique()));rng=np.random.default_rng(SELECTION_SEED+int(label));ids=ids[rng.permutation(len(ids))[:nclips]]
        for clip_id in ids:
            sub=g[g.clip_id.eq(clip_id)].sort_values('start_sample');pos=np.unique(np.linspace(0,len(sub)-1,min(SMOKE_WINDOWS_PER_CLIP,len(sub)),dtype=int));chunks.append(sub.iloc[pos])
    return pd.concat(chunks).sort_values(['clip_id','start_sample']).reset_index(drop=True)
active={}
for split in ['train','validation']:
    f=allowed_windows[allowed_windows.split.eq(split)].sort_values(['clip_id','start_sample']).reset_index(drop=True)
    active[split]=select_smoke(f,SMOKE_TRAIN_CLIPS_PER_CLASS if split=='train' else SMOKE_VAL_CLIPS_PER_CLASS) if RUN_MODE=='smoke' else f
    assert set(active[split].label_id)==set(LABELS) and not set(active[split].clip_id)&test_ids
selection_hash={s:digest(f[['window_id','clip_id','label_id','start_sample','end_sample']].to_dict('records')) for s,f in active.items()}
init_run(dict(stage=STAGE,mode=RUN_MODE,seeds=seeds,experiments=EXPERIMENTS,compute=compute,selection=selection_hash,selection_seed=SELECTION_SEED,
    smoke=[SMOKE_TRAIN_CLIPS_PER_CLASS,SMOKE_VAL_CLIPS_PER_CLASS,SMOKE_WINDOWS_PER_CLIP,SMOKE_BATCH_SIZE,SMOKE_EPOCHS,SMOKE_MAX_BATCHES],
    training=dict(batch=BATCH_SIZE,max_epochs=MAX_EPOCHS,patience=PATIENCE,lr=LEARNING_RATE,optimizer='Adam_default',select='validation_clip_macro_f1_earliest_tie'),
    frontend=dict(sr=8000,window=2400,nfft=256,hop=80,mel=40,mel_range=[0,4000],floor=PSD_FLOOR,filter='HTK_triangle_L1',rms_target=RMS_TARGET,max_rms_gain_db=MAX_RMS_GAIN_DB,
                  highpass_hz=HIGHPASS_HZ,highpass_order=HIGHPASS_ORDER,highpass='causal_sosfilt_whole_clip_zero_initial_state',standardization='train_per_mel_bin')))
for split,f in active.items():save_table(f.drop(columns=['path']),split+'_window_selection')
counts=save_table(pd.concat(active.values()).groupby(['species','split']).agg(clips=('clip_id','nunique'),windows=('window_id','size')).reset_index(),'selected_counts')
display(counts);print(DEVICE,compute,'| seeds:',seeds)

In [ ]:
verify_cell('persistence')
persistence_events=[]
def read_snapshot(key):
    if REMOTE is None:return None
    valid=[];seen=False
    for slot in range(2):
        folder=REMOTE/key;receipt=folder/f'slot{slot}.csv';archive=folder/f'slot{slot}.zip'
        if not receipt.exists() and not archive.exists():continue
        seen=True
        try:
            rec=pd.read_csv(receipt,keep_default_na=False);assert len(rec)==1;r=rec.iloc[0]
            assert r.run_id==RUN_ID and r['key']==key
            local=CACHE/f'{key}_slot{slot}.zip';shutil.copyfile(archive,local)
            assert local.stat().st_size==int(r['bytes']) and sha(local)==r.sha256
            with zipfile.ZipFile(local) as z:
                names=z.namelist();assert len(names)==len(set(names))
                inventory=pd.read_csv(io.BytesIO(z.read('snapshot_inventory.csv')))
                assert set(names)==set(inventory.path)|{'snapshot_inventory.csv'}
                for row in inventory.itertuples():
                    rel=archive_relative(row.path);assert rel.parts[0] in ('tables','models','figures') or row.path in ('report.html','source_notebook.ipynb')
                    payload=z.read(row.path);assert len(payload)==row.bytes and hashlib.sha256(payload).hexdigest()==row.sha256
            valid.append((int(r.generation),local))
        except (OSError,AssertionError,ValueError,KeyError,zipfile.BadZipFile) as e:
            persistence_events.append(dict(key=key,event='invalid_slot',detail=f'{slot}: {type(e).__name__}'))
    if seen and not valid:raise RuntimeError('Hai snapshot đều hỏng: '+key+'. Không tự ghi đè hoặc train lại.')
    return max(valid,key=lambda v:v[0]) if valid else None
def save_snapshot(key,files,progress):
    if REMOTE is None:return
    latest=read_snapshot(key);generation=(latest[0]+1) if latest else 1;slot=generation%2
    archive=CACHE/f'{key}_writing.zip';inventory=[]
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for path in sorted(set(map(Path,files))):
            rel=path.relative_to(RUN_OUT).as_posix();archive_relative(rel);payload=path.read_bytes()
            inventory.append(dict(path=rel,bytes=len(payload),sha256=hashlib.sha256(payload).hexdigest()));z.writestr(rel,payload)
        z.writestr('snapshot_inventory.csv',pd.DataFrame(inventory).to_csv(index=False))
    folder=REMOTE/key;folder.mkdir(parents=True,exist_ok=True);remote=folder/f'slot{slot}.zip'
    shutil.copyfile(archive,remote);assert sha(remote)==sha(archive)
    receipt=pd.DataFrame([dict(run_id=RUN_ID,key=key,generation=generation,progress=progress,sha256=sha(archive),bytes=archive.stat().st_size)])
    local=CACHE/f'{key}_receipt.csv';receipt.to_csv(local,index=False);shutil.copyfile(local,folder/f'slot{slot}.csv')
    reread=pd.read_csv(folder/f'slot{slot}.csv').iloc[0];assert reread.sha256==sha(archive) and int(reread.generation)==generation
    persistence_events.append(dict(key=key,event='saved',detail=f'generation={generation}, progress={progress}'))
def restore_snapshot(key):
    latest=read_snapshot(key)
    if latest is None:return False
    with zipfile.ZipFile(latest[1]) as z:
        inventory=pd.read_csv(io.BytesIO(z.read('snapshot_inventory.csv')))
        for r in inventory.itertuples():
            target=RUN_OUT.joinpath(*archive_relative(r.path).parts);assert target.resolve().is_relative_to(RUN_OUT.resolve())
            atomic_bytes(target,z.read(r.path))
    print('Đã phục hồi Drive:',key,flush=True);return True

## 3. Tạo log-mel theo từng frontend, rồi fit mean/std chỉ trên train
Loss có weight và balanced sampling dùng **cùng cache baseline**. RMS xử lý từng window. High-pass chạy causal một lần trên cả clip, trạng thái lọc bắt đầu 0 ở đầu clip, rồi mới lấy các window; không dùng zero-phase hoặc reset filter tại mỗi window overlap.

Cache là `.npy` trên ổ tạm, có hash/shape/identity. Loader train chỉ đọc cache, không giải mã WAV ngẫu nhiên ở từng batch. Nếu kernel ngắt lúc tạo cache, lần sau kiểm inventory và tạo lại phần chưa hợp lệ.

In [ ]:
verify_cell('feature_cache')
variants={'baseline':'baseline','rms_per_window':'rms_per_window','highpass_150':'highpass_150','weighted_ce':'baseline','balanced_sampler':'baseline'}
sos=signal.butter(HIGHPASS_ORDER,HIGHPASS_HZ,btype='highpass',fs=SAMPLE_RATE,output='sos')
def rms_normalize(x):
    rms=float(np.sqrt(np.mean(np.asarray(x,dtype='float64')**2)))
    gain=min(RMS_TARGET/max(rms,1e-8),10**(MAX_RMS_GAIN_DB/20))
    return np.asarray(x*gain,dtype='<f4')
def frontend_clip(x,variant):return signal.sosfilt(sos,np.asarray(x,dtype='float64')).astype('<f4') if variant=='highpass_150' else x
def prepare_mel_cache(variant):
    folder=CACHE/variant;folder.mkdir(exist_ok=True);inventory=folder/'inventory.csv'
    expected=['train_mel.npy','validation_mel.npy','statistics.npz']
    identity=digest(dict(run_id=RUN_ID,variant=variant,selection=selection_hash))
    valid=False
    if inventory.is_file():
        try:
            inv=pd.read_csv(inventory);valid=(set(inv.file)==set(expected) and inv.identity.eq(identity).all() and len(inv)==3 and
                all((folder/r.file).is_file() and sha(folder/r.file)==r.sha256 for r in inv.itertuples()))
        except (ValueError,OSError,KeyError):valid=False
    if not valid:
        needed=sum(len(f)*40*27*4 for f in active.values())
        if shutil.disk_usage(folder).free<needed+100_000_000:raise OSError('Không đủ ổ tạm cho cache log-mel')
        for split,frame in active.items():
            path=folder/(split+'_mel.npy');tmp=folder/(split+'_mel.writing.npy')
            arr=np.lib.format.open_memmap(tmp,mode='w+',dtype='<f4',shape=(len(frame),40,27))
            try:
                for k,(clip_id,g) in enumerate(frame.groupby('clip_id',sort=False),1):
                    x=frontend_clip(read_clip(clip_id),variant)
                    for index,r in g.iterrows():
                        w=x[int(r.start_sample):int(r.end_sample)]
                        if variant=='rms_per_window':w=rms_normalize(w)
                        arr[index]=logmel(w)
                    if k%200==0:print(variant,split,'clip',k,'/',frame.clip_id.nunique(),flush=True)
                arr.flush()
            finally:
                if getattr(arr,'_mmap',None) is not None:arr._mmap.close()
                del arr
            os.replace(tmp,path)
        tr=np.load(folder/'train_mel.npy',mmap_mode='r')
        mean=tr.mean(axis=(0,2),dtype=np.float64).astype('<f4');square=np.zeros(40,dtype='float64')
        for start in range(0,len(tr),1024):square+=np.square(np.asarray(tr[start:start+1024],dtype='float64')).sum(axis=(0,2))
        std=np.sqrt(np.maximum(square/(len(tr)*27)-mean.astype('float64')**2,1e-8)).astype('<f4')
        buffer=io.BytesIO();np.savez(buffer,mel_mean=mean,mel_std=std,fit_windows=len(tr),fit_frames=len(tr)*27)
        atomic_bytes(folder/'statistics.npz',buffer.getvalue());del tr
        atomic_bytes(inventory,pd.DataFrame([dict(file=f,sha256=sha(folder/f),identity=identity) for f in expected]).to_csv(index=False).encode())
    arrays={s:np.load(folder/(s+'_mel.npy'),mmap_mode='r') for s in active}
    with np.load(folder/'statistics.npz') as f:stats={k:f[k].copy() for k in f.files}
    assert int(stats['fit_windows'])==len(active['train']) and stats['mel_mean'].shape==(40,) and (stats['mel_std']>0).all()
    for split,arr in arrays.items():assert arr.shape==(len(active[split]),40,27) and np.isfinite(arr).all()
    return arrays,stats
feature_sets={v:prepare_mel_cache(v) for v in sorted({variants[e] for e in EXPERIMENTS})}
stat_rows=[]
for variant,(arrays,stats) in feature_sets.items():
    for i in range(40):stat_rows.append(dict(variant=variant,mel_bin=i,train_mean=stats['mel_mean'][i],train_std=stats['mel_std'][i],fit_windows=int(stats['fit_windows'])))
save_table(pd.DataFrame(stat_rows),'normalization_statistics')
check('Mọi thống kê fit đúng train',all(int(stats['fit_windows'])==len(active['train']) for arrays,stats in feature_sets.values()))
print('Cache sẵn sàng:',list(feature_sets),flush=True)

## 4. DS-CNN và cách cân bằng
DS-CNN sao chép đúng định nghĩa B (6.244 tham số), khởi tạo mới cho mỗi cấu hình/seed. Các cấu hình cùng seed bắt đầu cùng trọng số. Validation luôn giữ phân bố tự nhiên, không lấy mẫu cân bằng.

`weighted_ce`: weight lớp = N/(4×n_lớp), dùng số window train. `balanced_sampler`: weight từng window = 1/n_lớp; replacement=True, số mẫu mỗi epoch=N. Số lượt gặp mỗi lớp thực tế được ghi riêng. Hai phương pháp không bật đồng thời.

In [ ]:
verify_cell("models")
def conv2(cin,cout,k=3,stride=1,groups=1):
    return nn.Sequential(nn.Conv2d(cin,cout,k,stride,k//2,groups=groups,bias=False),
                         nn.BatchNorm2d(cout),nn.ReLU())

class DSCNN(nn.Module):
    def __init__(self):
        super().__init__(); layers=[conv2(1,32,stride=2)]
        for _ in range(4): layers.extend([conv2(32,32,groups=32),conv2(32,32,k=1)])
        self.features=nn.Sequential(*layers)
        self.head=nn.Sequential(nn.Dropout(.2),nn.Linear(32,4))
    def forward(self,x): return self.head(self.features(x).mean((2,3)))

class MelDataset(Dataset):
    def __init__(self,variant,split):
        assert split in ('train','validation')
        self.arrays,self.stats=feature_sets[variant];self.split=split;self.y=active[split].label_id.to_numpy()
    def __len__(self):return len(self.y)
    def __getitem__(self,index):
        x=(self.arrays[self.split][index]-self.stats['mel_mean'][:,None])/self.stats['mel_std'][:,None]
        return torch.from_numpy(np.array(x[None],dtype='float32',copy=True)),int(self.y[index])
train_counts=np.bincount(active['train'].label_id,minlength=4);assert (train_counts>0).all()
class_weights=torch.tensor(train_counts.sum()/(4*train_counts),dtype=torch.float32,device=DEVICE)
balance_table=save_table(pd.DataFrame({'species':LABEL_NAMES,'train_windows':train_counts,'natural_share':train_counts/train_counts.sum(),
    'ce_weight':class_weights.cpu().numpy(),'sampler_expected_share':np.full(4,.25)}),'balance_definition')
display(balance_table)
def make_train_loader(experiment,order_generator,sampler_generator):
    dataset=MelDataset(variants[experiment],'train');batch=SMOKE_BATCH_SIZE if RUN_MODE=='smoke' else BATCH_SIZE
    sampler=None
    if experiment=='balanced_sampler':
        weights=1/train_counts[dataset.y];sampler=WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),len(dataset),replacement=True,generator=sampler_generator)
    return DataLoader(dataset,batch_size=batch,shuffle=sampler is None,sampler=sampler,num_workers=0,drop_last=False,generator=order_generator)
@torch.inference_mode()
def predict(model,variant,split):
    loader=DataLoader(MelDataset(variant,split),batch_size=BATCH_SIZE,shuffle=False,num_workers=0,generator=torch.Generator().manual_seed(0))
    model.eval();output=[]
    for x,y in loader:output.append(torch.softmax(model(x.to(DEVICE)),dim=1).cpu().numpy())
    return np.concatenate(output)
def metrics_from_probabilities(frame,p):
    pred=p.argmax(1);precision,recall,f1,support=precision_recall_fscore_support(frame.label_id,pred,labels=LABELS,zero_division=0)
    return dict(macro_f1=float(f1.mean()),accuracy=float(pred.__eq__(frame.label_id.to_numpy()).mean()),balanced_accuracy=float(recall.mean()),precision=precision,recall=recall,f1=f1,support=support)
seed_all(seeds[0]);probe_model=DSCNN().to(DEVICE)
check('DS-CNN khớp B: 6244 tham số và 4 logits',sum(p.numel() for p in probe_model.parameters())==6244 and tuple(probe_model(torch.zeros(2,1,40,27,device=DEVICE)).shape)==(2,4))
del probe_model
fig,ax=plt.subplots(figsize=(9,3));balance_table.set_index('species')[['natural_share','sampler_expected_share']].plot.bar(ax=ax);ax.set_ylabel('Tỷ trọng window/lượt lấy mẫu');ax.tick_params(axis='x',rotation=15);save_figure('01_balance_design')
fig,axes=plt.subplots(2,len(feature_sets),figsize=(5*len(feature_sets),6),squeeze=False)
images=[]
for split in ['train','validation']:
    idx=int(np.flatnonzero(active[split].label_id.to_numpy()==1)[0])
    images.extend(feature_sets[v][0][split][idx] for v in feature_sets)
vmin,vmax=np.percentile(np.stack(images),[2,98])
for row,split in enumerate(['train','validation']):
    idx=int(np.flatnonzero(active[split].label_id.to_numpy()==1)[0])
    for col,variant in enumerate(feature_sets):
        arr=feature_sets[variant][0][split][idx];im=axes[row,col].imshow(arr,origin='lower',aspect='auto',cmap='magma',vmin=vmin,vmax=vmax)
        axes[row,col].set(title=f'{split}: {variant}',xlabel='Frame',ylabel='Mel bin');fig.colorbar(im,ax=axes[row,col])
save_figure('02_frontend_examples')

## 5. Huấn luyện, chọn checkpoint và tiếp tục khi mất runtime
Adam lr=0,001, CE, batch 64, max 40/patience 8 trong full. Hòa F1 giữ epoch sớm. Mỗi epoch log tiến độ batch, cả clip/window F1 và albopictus recall. Log loss train phản ánh objective đang dùng, không so sánh trực tiếp weighted loss với CE thường.

`last.pt` chứa model/optimizer, best weights, history và RNG Python/NumPy/Torch/CUDA + hai generator loader/sampler. Mỗi unit `cấu_hình_seed` lưu hai slot Drive có checksum; dừng nếu cả hai slot hỏng. Epoch đang dở chạy lại từ checkpoint epoch trước. Kết quả đã hoàn tất được nạp, không train lại. Chưa chứng minh tái lập bit-for-bit giữa các GPU/phiên bản khác nhau: run identity ngăn resume nhầm môi trường.

In [ ]:
verify_cell('training')
def rng_state(order,sampler):
    n=np.random.get_state()
    return dict(python=random.getstate(),numpy=[n[0],n[1].tolist(),n[2],n[3],n[4]],torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else [],order=order.get_state(),sampler=sampler.get_state())
def restore_rng(state,order,sampler):
    random.setstate(state['python']);n=state['numpy'];np.random.set_state((n[0],np.asarray(n[1],dtype=np.uint32),n[2],n[3],n[4]))
    torch.set_rng_state(state['torch'].cpu());order.set_state(state['order'].cpu());sampler.set_state(state['sampler'].cpu())
    if DEVICE.type=='cuda':
        assert len(state['cuda'])==torch.cuda.device_count();torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])
def torch_save_atomic(value,path):
    buffer=io.BytesIO();torch.save(value,buffer);atomic_bytes(path,buffer.getvalue())
def load_checkpoint(path,unit):
    payload=torch.load(path,map_location='cpu',weights_only=True)
    assert payload['format']==1 and payload['run_id']==RUN_ID and payload['unit']==unit and payload['mode']==RUN_MODE
    return payload
def train_one(experiment,seed):
    unit=f'{experiment}_seed{seed}';folder=RUN_OUT/'models'/unit;folder.mkdir(parents=True,exist_ok=True)
    last=folder/'last.pt';best_path=folder/'best.pt';history_path=TABLES/(unit+'_history.csv')
    if not last.is_file():restore_snapshot(unit)
    seed_all(seed);model=DSCNN().to(DEVICE);initial_sha=digest({k:hashlib.sha256(v.detach().cpu().numpy().tobytes()).hexdigest() for k,v in model.state_dict().items()})
    optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
    order=torch.Generator().manual_seed(seed);sampler_gen=torch.Generator().manual_seed(seed+1000)
    loader=make_train_loader(experiment,order,sampler_gen)
    history=[];best_weights=None;best_score=-1.;best_epoch=0;bad_epochs=0;completed=False;start_epoch=0;fit_seconds=0.
    if last.is_file():
        checkpoint=load_checkpoint(last,unit)
        assert checkpoint['initial_sha']==initial_sha and checkpoint['experiment']==experiment and checkpoint['seed']==seed
        model.load_state_dict(checkpoint['model']);optimizer.load_state_dict(checkpoint['optimizer']);restore_rng(checkpoint['rng'],order,sampler_gen)
        history=checkpoint['history'];start_epoch=checkpoint['next_epoch'];best_weights=checkpoint['best_weights'];best_score=checkpoint['best_score']
        best_epoch=checkpoint['best_epoch'];bad_epochs=checkpoint['bad_epochs'];completed=checkpoint['completed'];fit_seconds=checkpoint['fit_seconds']
        print(unit,'resume epoch',start_epoch,'completed=',completed,flush=True)
    epochs=SMOKE_EPOCHS if RUN_MODE=='smoke' else MAX_EPOCHS
    for epoch in range(start_epoch,epochs):
        if completed:break
        print(unit,f'epoch {epoch+1}/{epochs}: train bắt đầu',flush=True);model.train()
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        started=time.perf_counter();numerator=denominator=0.;seen=np.zeros(4,dtype=int);updates=0
        before=next(model.parameters()).detach().clone()
        for step,(x,y) in enumerate(loader):
            if RUN_MODE=='smoke' and step>=SMOKE_MAX_BATCHES:break
            optimizer.zero_grad(set_to_none=True);target=y.to(DEVICE);logits=model(x.to(DEVICE))
            weights=class_weights if experiment=='weighted_ce' else None
            loss=nn.functional.cross_entropy(logits,target,weight=weights);assert torch.isfinite(loss)
            loss.backward()
            if step==0:assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
            optimizer.step();seen+=np.bincount(y.numpy(),minlength=4);updates+=1
            batch_denom=float(weights[target].sum()) if weights is not None else len(y)
            numerator+=float(loss.detach())*batch_denom;denominator+=batch_denom
            if step==0 or (step+1)%100==0 or step+1==len(loader):print(unit,f'epoch {epoch+1}: batch {step+1}/{len(loader)}',flush=True)
        assert updates>0 and not torch.equal(before,next(model.parameters()).detach())
        p=predict(model,variants[experiment],'validation');clip=aggregate_clips(active['validation'],p)
        window_metrics=metrics_from_probabilities(active['validation'],p);clip_metrics=metrics_from_probabilities(clip,clip[[f'p{i}' for i in LABELS]].to_numpy())
        score=clip_metrics['macro_f1'];val_ce=float(-np.log(np.clip(p[np.arange(len(p)),active['validation'].label_id],1e-12,1)).mean())
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        seconds=time.perf_counter()-started;fit_seconds+=seconds
        row=dict(epoch=epoch+1,train_objective=numerator/denominator,validation_ce=val_ce,validation_clip_macro_f1=score,
            validation_window_macro_f1=window_metrics['macro_f1'],albopictus_clip_recall=float(clip_metrics['recall'][1]),
            albopictus_window_recall=float(window_metrics['recall'][1]),updates=updates,windows_seen=int(seen.sum()),seconds=seconds)
        row.update({f'seen_label{i}':int(seen[i]) for i in LABELS});history.append(row)
        if score>best_score+1e-12:
            best_score=score;best_epoch=epoch+1;bad_epochs=0;best_weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:bad_epochs+=1
        completed=epoch+1>=epochs or bad_epochs>=PATIENCE
        checkpoint=dict(format=1,run_id=RUN_ID,mode=RUN_MODE,unit=unit,experiment=experiment,seed=seed,initial_sha=initial_sha,
            model=model.state_dict(),optimizer=optimizer.state_dict(),best_weights=best_weights,best_score=best_score,best_epoch=best_epoch,
            bad_epochs=bad_epochs,next_epoch=epoch+1,completed=completed,fit_seconds=fit_seconds,history=history,rng=rng_state(order,sampler_gen))
        torch_save_atomic(checkpoint,last);save_table(pd.DataFrame(history),unit+'_history')
        if REMOTE is not None:print(unit,'đang lưu checkpoint Drive...',flush=True)
        save_snapshot(unit,[last,history_path,TABLES/'run_settings.csv',RUN_OUT/'source_notebook.ipynb'],epoch+1)
        print(unit,f'epoch {epoch+1}: clip F1={score:.4f}, window F1={window_metrics["macro_f1"]:.4f}, alb recall={clip_metrics["recall"][1]:.4f}',flush=True)
    assert best_weights is not None and history and completed
    assert [r['epoch'] for r in history]==list(range(1,len(history)+1))
    model.load_state_dict(best_weights);model.eval();_,stats=feature_sets[variants[experiment]]
    torch_save_atomic(dict(format=1,run_id=RUN_ID,mode=RUN_MODE,unit=unit,experiment=experiment,seed=seed,model=best_weights,
        selected_epoch=best_epoch,normalization={k:np.asarray(v).tolist() for k,v in stats.items()},label_names=LABEL_NAMES,
        rms_target=RMS_TARGET,max_rms_gain_db=MAX_RMS_GAIN_DB,highpass_hz=HIGHPASS_HZ,highpass_order=HIGHPASS_ORDER),best_path)
    restored=DSCNN().to(DEVICE);restored.load_state_dict(load_checkpoint(best_path,unit)['model']);restored.eval()
    reference=predict(model,variants[experiment],'validation');roundtrip=predict(restored,variants[experiment],'validation')
    check('Checkpoint roundtrip '+unit,np.array_equal(reference,roundtrip));del restored
    info=dict(experiment=experiment,seed=seed,unit=unit,epochs_run=len(history),best_epoch=best_epoch,updates=sum(r['updates'] for r in history),
        fit_seconds=fit_seconds,initial_sha=initial_sha,state=RUN_MODE+'_complete',selected_clip_macro_f1=best_score)
    return model,info,reference

## 6. Chạy các cấu hình và đánh giá checkpoint được chọn
Không nhập checkpoint 04 vào baseline 06. Baseline mới chạy cùng seed, kiến trúc, ngân sách và vòng lặp với các đối chứng; điểm 04 chỉ là tham chiếu lịch sử. Dự đoán train giúp nhận ra train tốt/validation kém, không dùng chọn checkpoint.

CSV dự đoán validation được lưu ở cả window/clip. Mọi clip có đủ window trong pool full, không chọn riêng window dễ. Khi smoke, bảng vẫn xuất để kiểm tra đường report nhưng không dùng đánh giá model.

In [ ]:
verify_cell('run_experiments')
metric_rows=[];class_rows=[];domain_rows=[];training_rows=[];histories={};confidences=[]
validation_predictions={}
def record(experiment,seed,split,p):
    frame=active[split];c=aggregate_clips(frame,p)
    for level,f,prob in [('window',frame,p),('clip',c,c[[f'p{i}' for i in LABELS]].to_numpy())]:
        m=metrics_from_probabilities(f,prob)
        metric_rows.append(dict(mode=RUN_MODE,experiment=experiment,seed=seed,split=split,level=level,examples=len(f),
            macro_f1=m['macro_f1'],accuracy=m['accuracy'],balanced_accuracy=m['balanced_accuracy']))
        for i in LABELS:class_rows.append(dict(mode=RUN_MODE,experiment=experiment,seed=seed,split=split,level=level,label_id=int(i),species=LABEL_NAMES[i],
            precision=float(m['precision'][i]),recall=float(m['recall'][i]),f1=float(m['f1'][i]),support=int(m['support'][i])))
    for (species,domain),g in c.groupby(['species','domain']):domain_rows.append(dict(experiment=experiment,seed=seed,split=split,species=species,domain=domain,
        clips=len(g),recall=float(g.prediction.eq(g.label_id).mean())))
    if split=='validation':
        prefix=f'{experiment}_seed{seed}';c=c.assign(experiment=experiment,seed=seed,mode=RUN_MODE)
        w=frame[['window_id','clip_id','label_id','domain']].copy();w['prediction']=p.argmax(1)
        for i in LABELS:w[f'p{i}']=p[:,i]
        save_table(w.assign(experiment=experiment,seed=seed,mode=RUN_MODE),prefix+'_validation_windows');save_table(c,prefix+'_validation_clips')
        validation_predictions[(experiment,seed)]=c
        alb=c[c.label_id.eq(1)];maxp=alb[[f'p{i}' for i in LABELS]].max(axis=1)
        confidences.append(dict(experiment=experiment,seed=seed,alb_clips=len(alb),mean_p_albopictus=alb.p1.mean(),mean_p_minimus=alb.p3.mean(),
            alb_wrong_score_over95=int((alb.prediction.ne(1)&maxp.gt(.95)).sum()),alb_correct=int(alb.prediction.eq(1).sum())))
for seed in seeds:
    for experiment in EXPERIMENTS:
        print('\nBắt đầu:',experiment,'seed',seed,'mode',RUN_MODE,flush=True)
        model,info,val_p=train_one(experiment,seed);training_rows.append(info)
        histories[(experiment,seed)]=pd.read_csv(TABLES/(info['unit']+'_history.csv'))
        record(experiment,seed,'validation',val_p)
        record(experiment,seed,'train',predict(model,variants[experiment],'train'))
        for name,rows in [('metrics',metric_rows),('per_class_metrics',class_rows),('species_domain_recall',domain_rows),('training_summary',training_rows),('albopictus_confidence',confidences)]:save_table(pd.DataFrame(rows),name)
        unit_files=list((RUN_OUT/'models'/info['unit']).glob('*.pt'))+list(TABLES.glob(info['unit']+'*.csv'))+[TABLES/'run_settings.csv',RUN_OUT/'source_notebook.ipynb']
        save_snapshot(info['unit'],unit_files,info['epochs_run'])
        del model
metrics=pd.DataFrame(metric_rows);per_class=pd.DataFrame(class_rows);training_summary=pd.DataFrame(training_rows)
display(training_summary)
if RUN_MODE=='smoke':display(Markdown('**SMOKE hoàn tất — các điểm dưới đây chỉ kiểm code, không dùng xếp hạng hoặc kết luận nguyên nhân.**'))

## 7. So sánh cùng seed, xem tác động lên cả bốn loài
Báo clip/window macro-F1 và albopictus recall. Delta tính so với baseline **cùng seed của 06**, không lấy hiệu trực tiếp với 04. Trung bình/độ lệch chuẩn giữa seed thể hiện biến thiên khởi tạo/huấn luyện, không giải quyết hạn chế chỉ có một nguồn albopictus validation.

Cải thiện albopictus nhưng làm các loài khác giảm mạnh cần được nhìn thấy. Không chọn cấu hình chỉ dựa trên recall một lớp; không dùng số window overlap để tạo khoảng tin cậy giả.

In [ ]:
verify_cell('comparison')
val_metrics=metrics[metrics.split.eq('validation')].copy();delta_rows=[]
for r in val_metrics.itertuples():
    base=val_metrics[(val_metrics.experiment=='baseline')&(val_metrics.seed==r.seed)&(val_metrics.level==r.level)].iloc[0]
    current=per_class[(per_class.experiment==r.experiment)&(per_class.seed==r.seed)&(per_class.split=='validation')&(per_class.level==r.level)].set_index('label_id')
    control=per_class[(per_class.experiment=='baseline')&(per_class.seed==r.seed)&(per_class.split=='validation')&(per_class.level==r.level)].set_index('label_id')
    row=dict(experiment=r.experiment,seed=r.seed,level=r.level,macro_f1=r.macro_f1,delta_macro_f1=r.macro_f1-base.macro_f1,
        albopictus_recall=current.loc[1,'recall'],delta_albopictus_recall=current.loc[1,'recall']-control.loc[1,'recall'])
    for i,name in [(0,'aegypti'),(2,'dirus'),(3,'minimus')]:row['delta_'+name+'_recall']=current.loc[i,'recall']-control.loc[i,'recall']
    delta_rows.append(row)
paired=save_table(pd.DataFrame(delta_rows),'paired_seed_deltas');display(paired)
columns=['macro_f1','delta_macro_f1','albopictus_recall','delta_albopictus_recall','delta_aegypti_recall','delta_dirus_recall','delta_minimus_recall']
summary=paired.groupby(['experiment','level'],sort=False)[columns].agg(['mean','std','count'])
summary.columns=['_'.join(c) for c in summary.columns];summary=save_table(summary.reset_index(),'seed_summary');display(summary)
fig,axes=plt.subplots(1,2,figsize=(13,4))
for ax,level in zip(axes,['clip','window']):
    sub=paired[paired.level.eq(level)]
    for seed,g in sub.groupby('seed'):
        g=g.set_index('experiment').loc[EXPERIMENTS];ax.plot(np.arange(len(EXPERIMENTS)),g.macro_f1,'o-',alpha=.7,label=f'seed {seed}')
    ax.set(xticks=np.arange(len(EXPERIMENTS)),xticklabels=EXPERIMENTS,ylabel='Validation macro-F1',title=f'{level} — '+('SMOKE, không xếp hạng' if RUN_MODE=='smoke' else 'cùng seed'));ax.tick_params(axis='x',rotation=20);ax.legend()
save_figure('03_paired_scores')
fig,axes=plt.subplots(1,4,figsize=(17,4),sharey=True)
for label,ax in enumerate(axes):
    for seed in seeds:
        g=per_class[(per_class.split=='validation')&(per_class.level=='clip')&(per_class.label_id==label)&(per_class.seed==seed)].set_index('experiment').loc[EXPERIMENTS]
        ax.plot(np.arange(len(EXPERIMENTS)),g.recall,'o-',alpha=.7,label=f'seed {seed}')
    ax.set(xticks=np.arange(len(EXPERIMENTS)),xticklabels=EXPERIMENTS,ylim=(-.03,1.03),title=LABEL_NAMES[label],ylabel='Clip recall');ax.tick_params(axis='x',rotation=30)
save_figure('04_per_species_recall')
fig,axes=plt.subplots(2,3,figsize=(15,8))
for ax,experiment in zip(axes.ravel(),EXPERIMENTS):
    for seed in seeds:
        h=histories[(experiment,seed)];ax.plot(h.epoch,h.validation_clip_macro_f1,label=f'clip / {seed}');ax.plot(h.epoch,h.validation_window_macro_f1,'--',alpha=.7,label=f'window / {seed}')
    ax.set(title=experiment,xlabel='Epoch',ylabel='Validation macro-F1');ax.legend(fontsize=7)
for ax in axes.ravel()[len(EXPERIMENTS):]:ax.set_visible(False)
save_figure('05_learning_curves')
fig,axes=plt.subplots(2,3,figsize=(14,8));short=['aegypti','albopictus','dirus','minimus']
for ax,experiment in zip(axes.ravel(),EXPERIMENTS):
    c=validation_predictions[(experiment,seeds[0])];sns.heatmap(confusion_matrix(c.label_id,c.prediction,labels=LABELS),annot=True,fmt='d',cbar=False,cmap='Blues',ax=ax,xticklabels=short,yticklabels=short)
    ax.set_title(f'{experiment}, seed {seeds[0]}');ax.set_xlabel('Dự đoán');ax.set_ylabel('Nhãn thật');ax.tick_params(axis='x',rotation=20)
for ax in axes.ravel()[len(EXPERIMENTS):]:ax.set_visible(False)
save_figure('06_confusion_first_seed')

## 8. Kiểm tra và report
Kiểm tín hiệu tổng hợp cho frontend, chống truy cập test, cập nhật trọng số, roundtrip checkpoint, cùng khởi tạo giữa các cấu hình cùng seed, số update smoke và dữ liệu A/B không đổi. Các kiểm tra chạy được trên CPU; CUDA/Drive cần xác nhận thêm trên runtime thực của bạn.

Cách diễn giải sau full:
- RMS cải thiện: mức âm có ảnh hưởng trong recipe này; không khẳng định chỉ do âm lượng.
- High-pass cải thiện: dải thấp có ảnh hưởng; cần xem các loài khác có mất thông tin không.
- Weight/sampler cải thiện: phân bố đóng góp lớp có ảnh hưởng, chưa sửa thiếu đa dạng nguồn.
- Không cải thiện: chưa đủ để quy lỗi riêng cho kiến trúc. Quay lại kiểm nhãn, nguồn, độ rõ tiếng muỗi và độ phủ dữ liệu trong 05.
- Kết quả tốt trên nguồn D1 hiện tại vẫn chưa chứng minh khái quát sang microphone INMP441.

In [ ]:
def write_report(title,narrative,table_names):
    sections=[f'<h1>{html.escape(title)}</h1>',f'<p>Mode: <b>{RUN_MODE}</b> | Run: {RUN_ID[:16]} | {datetime.now().isoformat(timespec="seconds")}</p>',narrative]
    for name in table_names:
        p=TABLES/(name+'.csv')
        if p.exists():
            sections.extend([f'<h2>{html.escape(name)}</h2>','<div class="scroll">'+pd.read_csv(p).to_html(index=False,border=0,float_format=lambda x:f'{x:.4f}')+'</div>'])
    for p in sorted(FIGURES.glob('*.png')):sections.append(f'<h2>{html.escape(p.stem)}</h2><img src="figures/{p.name}" alt="{html.escape(p.stem)}">')
    text='<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>'+html.escape(title)+'</title><style>body{font:16px/1.55 system-ui;max-width:1200px;margin:32px auto;padding:0 20px;color:#172a3a}table{border-collapse:collapse;font-size:13px}td,th{padding:7px;border-bottom:1px solid #ddd;text-align:left}th{background:#edf3f7}.scroll{overflow:auto}img{max-width:100%;height:auto}h2{margin-top:32px}p{max-width:1000px}</style>'+''.join(sections)+'</html>'
    atomic_bytes(RUN_OUT/'report.html',text.encode('utf-8'))
    print('Report:',RUN_OUT/'report.html')

verify_cell("report_helpers")

In [ ]:
verify_cell('final_checks')
t=np.arange(2400)/8000;tone=.1*np.sin(2*np.pi*500*t)
check('Log-mel đúng shape và gain x2 thêm 6.02dB',logmel(tone).shape==(40,27) and np.allclose((logmel(2*tone)-logmel(tone))[logmel(tone)>-100],20*np.log10(2),atol=.02))
check('RMS đạt mục tiêu khi không chạm gain cap',np.isclose(np.sqrt(np.mean(rms_normalize(tone).astype('float64')**2)),RMS_TARGET,rtol=1e-5))
check('RMS silence hữu hạn',np.isfinite(rms_normalize(np.zeros(2400))).all() and np.array_equal(rms_normalize(np.zeros(2400)),np.zeros(2400)))
_,response=signal.sosfreqz(sos,worN=np.array([50.,500.]),fs=8000)
check('High-pass 150 Hz suy giảm 50Hz mạnh hơn 500Hz',abs(response[0])<.2 and abs(response[1])>.95)
long=np.sin(2*np.pi*100*np.arange(8000)/8000);whole=signal.sosfilt(sos,long);first,state=signal.sosfilt(sos,long[:2400],zi=np.zeros((len(sos),2)));second,_=signal.sosfilt(sos,long[2400:],zi=state)
check('Filter causal nối chunk khớp chạy toàn clip',np.allclose(np.r_[first,second],whole))
check('Cùng seed khởi tạo cùng trọng số',training_summary.groupby('seed').initial_sha.nunique().eq(1).all())
check('Đủ cấu hình/seed',len(training_summary)==len(EXPERIMENTS)*len(seeds) and training_summary.state.eq(RUN_MODE+'_complete').all())
check('Mọi unit có update',training_summary.updates.gt(0).all())
for (experiment,seed),c in validation_predictions.items():
    check('Validation giữ phân bố '+experiment+str(seed),set(c.clip_id)==set(active['validation'].clip_id) and len(c)==active['validation'].clip_id.nunique())
    actual=f1_score(c.label_id,c.prediction,labels=LABELS,average='macro',zero_division=0)
    recorded=metrics.query('experiment==@experiment and seed==@seed and split=="validation" and level=="clip"').macro_f1.item()
    check('Metric clip tính lại '+experiment+str(seed),np.isclose(actual,recorded))
if RUN_MODE=='smoke':check('Smoke đúng ngân sách',training_summary.epochs_run.le(SMOKE_EPOCHS).all() and training_summary.updates.le(SMOKE_EPOCHS*SMOKE_MAX_BATCHES).all())
for p,before in PROTECTED.items():check('Không sửa '+str(p.relative_to(ROOT)),sha(p)==before)
check('Không đọc audio test',not accessed&test_ids and accessed<=allowed_ids)
check('Các cell chạy khớp source trên kernel',set(core_cells)<=verified_tags)
save_table(pd.DataFrame(checks),'validation_checks');save_table(pd.DataFrame([dict(clip_id=i,split=clip_lookup.loc[i,'split']) for i in sorted(accessed)]),'audio_access_log')
save_table(pd.DataFrame(persistence_events,columns=['key','event','detail']),'persistence_events')
write_report('B06 — Thử nghiệm đối chứng lỗi albopictus',
    '<p>'+('SMOKE: chỉ kiểm code, không dùng điểm để kết luận hoặc chọn model. ' if RUN_MODE=='smoke' else 'FULL: delta so baseline cùng seed, không so trực tiếp điểm với lượt 04. ')+
    'DS-CNN cố định; một thay đổi mỗi cấu hình. Mean/std fit train từng frontend. Checkpoint chọn bằng clip macro-F1, báo thêm window macro-F1 và recall từng loài. '+
    'Một nguồn albopictus D1 validation không đủ chứng minh khái quát ra thiết bị mới. Không đọc audio test. Thời gian fit không bao gồm lưu Drive và chưa phải latency ESP32.</p>',
    ['selected_counts','balance_definition','training_summary','seed_summary','paired_seed_deltas','per_class_metrics','albopictus_confidence','validation_checks'])
report_files=[RUN_OUT/'report.html',RUN_OUT/'source_notebook.ipynb',*TABLES.glob('*.csv'),*FIGURES.glob('*.png')]
save_snapshot('report',report_files,len(training_summary))
display(pd.DataFrame(checks));print('PASS:',len(checks),'| Report:',RUN_OUT/'report.html')